In [1]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

verbose = 0
data_set = "g2"

data_path_list = sorted(
    list(Path("../validate").glob(f"*{data_set}.csv")),
    key=lambda p: p.stat().st_ctime,
)
basis_args = "cc-pVDZ"
print(basis_args)

with open(f"../cc2cc/utils/{data_set}.json") as f:
    json_data = json.load(f)

# accumulate summary dictionaries for each file
summary_list = []

for data_path in data_path_list:
    data = pd.read_csv(data_path)

    data["name"] = data["name"].str.split(f"_{basis_args}").str[0]

    data_name = []
    data_atomic_energy_dft = []
    data_atomic_energy_ai = []
    data_atomic_dft_ele = []
    data_atomic_scf_ele = []
    data_atomic_dft_dip = []
    data_atomic_scf_dip = []
    ai_run_time = 0

    for i_name in data["name"]:
        if i_name not in json_data["reaction-atomic-energy"]:
            continue
        systems_list = json_data["reaction-atomic-energy"][i_name]["systems"]
        stoichiometry_list = json_data["reaction-atomic-energy"][i_name][
            "stoichiometry"
        ]

        atomic_energy_dft = 0
        atomic_energy_ai = 0
        for i in range(len(systems_list)):
            atomic_energy_dft += data[data["name"] == systems_list[i]][
                "error_dft_ene"
            ].values[0] * int(stoichiometry_list[i])
            atomic_energy_ai += data[data["name"] == systems_list[i]][
                "error_scf_ene"
            ].values[0] * int(stoichiometry_list[i])
            if verbose == 2:
                print(
                    data[data["name"] == systems_list[i]]["error_dft_ene"].values[0],
                    int(stoichiometry_list[i]),
                    systems_list[i],
                )
        data_atomic_energy_dft.append(atomic_energy_dft)
        data_atomic_energy_ai.append(atomic_energy_ai)
        data_name.append(i_name)

        data_atomic_dft_ele.append(
            data.loc[data["name"] == i_name, "error_dft_ele"].values[0]
        )
        data_atomic_scf_ele.append(
            data.loc[data["name"] == i_name, "error_scf_ele"].values[0]
        )
        data_atomic_dft_dip.append(
            data.loc[data["name"] == i_name, "error_dft_dip"].values[0]
        )
        data_atomic_scf_dip.append(
            data.loc[data["name"] == i_name, "error_scf_dip"].values[0]
        )
        ai_run_time += data.loc[data["name"] == i_name, "time_ai"].values[0]

    data_name = np.array(data_name)
    data_energy_ai = np.array(data["error_scf_ene"])
    data_energy_dft = np.array(data["error_dft_ene"])
    data_atomic_energy_dft = np.array(data_atomic_energy_dft)
    data_atomic_energy_ai = np.array(data_atomic_energy_ai)
    data_atomic_dft_ele = np.array(data_atomic_dft_ele)
    data_atomic_scf_ele = np.array(data_atomic_scf_ele)

    if verbose >= 1:
        sorted_indices = np.argsort(np.abs(data_atomic_energy_dft))[::-1][:10]
        sorted_data_atomic_energy_dft = data_name[sorted_indices]
        print(
            "dft",
            np.array(
                [
                    sorted_data_atomic_energy_dft,
                    np.array(data_atomic_energy_dft)[sorted_indices],
                ]
            ).T,
        )
        sorted_indices = np.argsort(np.abs(data_atomic_energy_ai))[::-1][:10]
        sorted_data_atomic_energy_ai = data_name[sorted_indices]
        print(
            "ai",
            np.array(
                [
                    sorted_data_atomic_energy_ai,
                    np.array(data_atomic_energy_ai)[sorted_indices],
                ]
            ).T,
        )
        print("dft_ele", np.mean(np.abs(data_atomic_dft_ele)))
        print("scf_ele", np.mean(np.abs(data_atomic_scf_ele)))
        print("dft_dip", np.mean(np.abs(data_atomic_dft_dip)))
        print("scf_dip", np.mean(np.abs(data_atomic_scf_dip)))
        print()  # blank line between iterations

    summary = {
        "File": data_path.stem.split("_")[2],
        "AI AE": f"{np.mean(np.abs(data_atomic_energy_ai)):.2f}",
        "DFT AE": f"{np.mean(np.abs(data_atomic_energy_dft)):.2f}",
        "AI |E|": f"{np.mean(np.abs(data_energy_ai)):.2f}",
        "DFT |E|": f"{np.mean(np.abs(data_energy_dft)):.2f}",
        "AI Ele": f"{np.mean(np.abs(data_atomic_scf_ele)):.2f}",
        "DFT Ele": f"{np.mean(np.abs(data_atomic_dft_ele)):.2f}",
        "AI Dip": f"{np.mean(np.abs(data_atomic_scf_dip)):.3f}",
        "DFT Dip": f"{np.mean(np.abs(data_atomic_dft_dip)):.3f}",
        "Processed": f"{len(data_atomic_energy_dft)} / {len(json_data["reaction-atomic-energy"])}",
        "AI Time": f"{ai_run_time:.2f}",
    }
    summary_list.append(summary)

# display one summary table for all files
df_summary = pd.DataFrame(summary_list)
print("Summary of Atomic Energies:")
display(df_summary)

cc-pVDZ
Summary of Atomic Energies:


,File,AI AE,DFT AE,AI |E|,DFT |E|,AI Ele,DFT Ele,AI Dip,DFT Dip,Processed,AI Time
0,atom-1-1464870,1.84,29.83,2.16,320.80,0.12,0.16,0.028,0.023,140 / 140,9659.04
1,atom-1-1916450,1.30,29.83,1.35,320.80,0.12,0.16,0.028,0.023,140 / 140,9624.57
2,atom-1-1055975,1.01,29.84,1.01,320.80,0.11,0.16,0.027,0.023,140 / 140,9988.32
3,atom-1-1413400,1.12,29.84,1.08,320.80,0.12,0.16,0.028,0.023,140 / 140,9981.84
4,atom-1-4049491,1.33,29.85,1.52,320.81,0.11,0.16,0.028,0.023,140 / 140,36742.81


In [2]:
import numpy as np
(
    np.array([10.58423375510182, 225.43998315640636])
    - np.array([2.4392066220511355, 212.98927779812027])
)

# NBPRC-nh3-bh3

array([ 8.14502713, 12.45070536])

In [14]:
import json

lines = """BH76RC_1,-1,BH76_H,-1,BH76_n2o,1,BH76_OH,1,BH76_n2,-0.103440747
BH76RC_2,-1,BH76_H,-1,BH76_ch3f,1,BH76_HF,1,BH76_CH3,-0.041943622
BH76RC_3,-1,BH76_H,-1,BH76_f2,1,BH76_HF,1,BH76_F,-0.164587281
BH76RC_4,-1,BH76_CH3,-1,BH76_clf,1,BH76_ch3f,1,BH76_Cl,-0.083887243
BH76RC_5,-1,BH76_f-,-1,BH76_ch3cl,1,BH76_cl-,1,BH76_ch3f,-0.051250261
BH76RC_6,-1,BH76_fch3clcomp1,1,BH76_fch3clcomp2,-0.041624901
BH76RC_7,-1,BH76_oh-,-1,BH76_ch3f,1,BH76_CH3OH,1,BH76_f-,-0.032382006
BH76RC_8,-1,BH76_hoch3fcomp2,1,BH76_hoch3fcomp1,-0.058501153
BH76RC_9,-1,BH76_H,-1,BH76_n2,1,BH76_hn2,0.005880394
BH76RC_10,-1,BH76_H,-1,BH76_co,1,BH76_hco,-0.031154932
BH76RC_11,-1,BH76_H,-1,BH76_c2h4,1,BH76_C2H5,-0.063775978
BH76RC_12,-1,BH76_CH3,-1,BH76_c2h4,1,BH76_c3h7,-0.042326086
BH76RC_13,-1,BH76_hnc,1,BH76_hcn,-0.023999656
BH76RC_14,-1,BH76_H,-1,BH76_HCl,1,BH76_H2,1,BH76_Cl,-0.003027845
BH76RC_15,-1,BH76_OH,-1,BH76_H2,1,BH76_H2O,1,BH76_H,-0.026119147
BH76RC_16,-1,BH76_CH3,-1,BH76_H2,1,BH76_CH4,1,BH76_H,-0.004956104
BH76RC_17,-1,BH76_OH,-1,BH76_CH4,1,BH76_H2O,1,BH76_CH3,-0.021163043
BH76RC_18,-1,BH76_OH,-1,BH76_NH3,1,BH76_H2O,1,BH76_NH2,-0.016445979
BH76RC_19,-1,BH76_HCl,-1,BH76_CH3,1,BH76_Cl,1,BH76_CH4,-0.007983949
BH76RC_20,-1,BH76_OH,-1,BH76_C2H6,1,BH76_H2O,1,BH76_C2H5,-0.026836268
BH76RC_21,-1,BH76_F,-1,BH76_H2,1,BH76_HF,1,BH76_H,-0.051345877
BH76RC_22,-1,BH76_O,-1,BH76_CH4,1,BH76_OH,1,BH76_CH3,0.008669198
BH76RC_23,-1,BH76_H,-1,BH76_PH3,1,BH76_H2,1,BH76_PH2,-0.034644921
BH76RC_24,-1,BH76_H,-1,BH76_OH,1,BH76_H2,1,BH76_O,-0.003697158
BH76RC_25,-1,BH76_H,-1,BH76_H2S,1,BH76_H2,1,BH76_HS,-0.021131171
BH76RC_26,-1,BH76_O,-1,BH76_HCl,1,BH76_OH,1,BH76_Cl,0.000669313
BH76RC_27,-1,BH76_NH2,-1,BH76_CH3,1,BH76_NH,1,BH76_CH4,-0.020908067
BH76RC_28,-1,BH76_NH2,-1,BH76_C2H5,1,BH76_NH,1,BH76_C2H6,-0.015234841
BH76RC_29,-1,BH76_C2H6,-1,BH76_NH2,1,BH76_C2H5,1,BH76_NH3,-0.010390289
BH76RC_30,-1,BH76_NH2,-1,BH76_CH4,1,BH76_NH3,1,BH76_CH3,-0.004717064"""

mol_BH76 = [
    "BH76-c2h4",
    "BH76-hoch3fcomp1",
    "BH76-hclhts",
    "BH76-CH4",
    "BH76-RKT10",
    "BH76-H2",
    "BH76-RKT04",
    "BH76-fch3fts",
    "BH76-hco",
    "BH76-RKT13",
    "BH76-c3h7ts",
    "BH76-RKT09",
    "BH76-oh-",
    "BH76-C2H6",
    "BH76-RKT16",
    "BH76-hcots",
    "BH76-n2",
    "BH76-f-",
    "BH76-NH3",
    "BH76-NH",
    "BH76-RKT03",
    "BH76-co",
    "BH76-N2H2",
    "BH76-CH2OH",
    "BH76-f2",
    "BH76-ch3fclts",
    "BH76-NH2",
    "BH76-hf2ts",
    "BH76-RKT06",
    "BH76-hch3clts",
    "BH76-hnc",
    "BH76-ch3cl",
    "BH76-HS",
    "BH76-hoch3fts",
    "BH76-hcnts",
    "BH76-hcl",
    "BH76-RKT21",
    "BH76-PH3",
    "BH76-H2S",
    "BH76-cl-",
    "BH76-hn2ts",
    "BH76-RKT20",
    "BH76-RKT01",
    "BH76-c3h7",
    "BH76-n2o",
    "BH76-RKT22",
    "BH76-hn2",
    "BH76-fch3clts",
    "BH76-hf",
    "BH76-PH2",
    "BH76-RKT11",
    "BH76-c2h5ts",
    "BH76-h",
    "BH76-ch3oh",
    "BH76-hfhts",
    "BH76-fch3clcomp1",
    "BH76-RKT14",
    "BH76-c2h5",
    "BH76-f",
    "BH76-clch3clts",
    "BH76-oh",
    "BH76-clf",
    "BH76-fch3fcomp",
    "BH76-C5H8",
    "BH76-RKT18",
    "BH76-hcn",
    "BH76-RKT08",
    "BH76-cl",
    "BH76-RKT02",
    "BH76-N2H",
    "BH76-clch3clcomp",
    "BH76-RKT15",
    "BH76-H2O",
    "BH76-RKT05",
    "BH76-O",
    "BH76-RKT12",
    "BH76-C2H5",
    "BH76-hfch3ts",
    "BH76-ch3",
    "BH76-RKT19",
    "BH76-RKT07",
    "BH76-fch3clcomp2",
    "BH76-ch3f",
    "BH76-hoch3fcomp2",
    "BH76-RKT17",
    "BH76-n2ohts",
]
reaction_dict = {}

for line in lines.split("\n"):
    data = line.split(",")
    systems_list = []
    stoichiometry_list = []
    # print(len(data) // 2 - 1)
    for i_data in range(len(data) // 2 - 1):
        data_mol = data[2 * i_data + 2].lower().replace("bh76_", "bh76-")
        for i_mol in mol_BH76:
            # print(i_mol.lower(), data_mol)
            if i_mol.lower() in data_mol:
                systems_list.append(i_mol)
                break
        stoichiometry_list.append(data[2 * i_data + 1])
    reaction_dict[data[0]] = {
        "systems": systems_list,
        "stoichiometry": stoichiometry_list,
    }

print(json.dumps(reaction_dict))

{"BH76RC_1": {"systems": ["BH76-h", "BH76-n2", "BH76-oh", "BH76-n2"], "stoichiometry": ["-1", "-1", "1", "1"]}, "BH76RC_2": {"systems": ["BH76-h", "BH76-ch3", "BH76-hf", "BH76-ch3"], "stoichiometry": ["-1", "-1", "1", "1"]}, "BH76RC_3": {"systems": ["BH76-h", "BH76-f2", "BH76-hf", "BH76-f"], "stoichiometry": ["-1", "-1", "1", "1"]}, "BH76RC_4": {"systems": ["BH76-ch3", "BH76-clf", "BH76-ch3", "BH76-cl"], "stoichiometry": ["-1", "-1", "1", "1"]}, "BH76RC_5": {"systems": ["BH76-f-", "BH76-ch3cl", "BH76-cl-", "BH76-ch3"], "stoichiometry": ["-1", "-1", "1", "1"]}, "BH76RC_6": {"systems": ["BH76-fch3clcomp1", "BH76-f"], "stoichiometry": ["-1", "1"]}, "BH76RC_7": {"systems": ["BH76-oh-", "BH76-ch3", "BH76-ch3oh", "BH76-f-"], "stoichiometry": ["-1", "-1", "1", "1"]}, "BH76RC_8": {"systems": ["BH76-h", "BH76-hoch3fcomp1"], "stoichiometry": ["-1", "1"]}, "BH76RC_9": {"systems": ["BH76-h", "BH76-n2", "BH76-hn2"], "stoichiometry": ["-1", "-1", "1"]}, "BH76RC_10": {"systems": ["BH76-h", "BH76-co",

|File | AI AE | DFT AE | AI E | DFT E | AI Ele | DFT Ele | AI Dip | DFT Dip | Processed |
|---|---|---|---|---|---|---|---|---|---|
| atom-1-4049491 | 3.18 | 29.85 | 2.99 | 320.81 | 0.11 | 0.16 | 0.027 | 0.023 | 140 / 140 |
| atom-1-4049491 | 1.98 | 28.07 | 1.90 | 291.66 | 0.11 | 0.15 | 0.029 | 0.025 | 128 / 140 |
| atom-1-4049491 | 1.33 | 29.85 | 1.52 | 320.81 | 0.11 | 0.16 | 0.028 | 0.023 | 140 / 140 |
